## 1. Imports & Setup

In [ ]:
import os
import sys
import json
import copy
import math
import warnings
import random
from collections import Counter
import numpy as np
import pandas as pd
import torch
from scipy.spatial import distance
from datasets import load_dataset
from tqdm.notebook import tqdm
from typing import List, Tuple, Dict, Optional

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device: CPU (AWDRNNModel not MPS compatible) ───────────────────
DEVICE = torch.device('cpu')
print(f'Using device: {DEVICE}')

# ── Paths ─────────────────────────────────────────────────────────
NOTEBOOK_DIR  = os.getcwd()
PROJECT_ROOT  = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
NASBENCH_PATH = os.path.join(PROJECT_ROOT, 'nas-bench-nlp-release')
RESULTS_PATH  = os.path.join(PROJECT_ROOT, 'results')

sys.path.insert(0, NASBENCH_PATH)
os.makedirs(RESULTS_PATH, exist_ok=True)

print(f'Project root  : {PROJECT_ROOT}')
print(f'NASBench path : {NASBENCH_PATH}')
print(f'Results path  : {RESULTS_PATH}')

# ── Import NAS-Bench-NLP modules ──────────────────────────────────
try:
    from model import AWDRNNModel
    from nas_environment import Environment
    print('NAS-Bench-NLP modules loaded successfully')
except ImportError as e:
    print(f'ERROR: {e}')
    raise

## 2. Load SOLD Data & Vocabulary

In [ ]:
with open(os.path.join(RESULTS_PATH, 'vocab.json'), 'r', encoding='utf-8') as f:
    word2idx = json.load(f)

VOCAB_SIZE = len(word2idx)
PAD_IDX    = word2idx['<PAD>']
UNK_IDX    = word2idx['<UNK>']
print(f'Vocabulary loaded: {VOCAB_SIZE} tokens')

print('Loading SOLD train split...')
sold_train  = load_dataset('sinhala-nlp/SOLD', split='train')
train_texts = sold_train['text']
print(f'Loaded {len(train_texts)} training samples')

## 3. Prepare Fixed SOLD Inputs for Hidden Covariance

In [ ]:
SEQ_LEN    = 35    # max token length per tweet
N_SEGMENTS = 64    # v9: 64 tweets (up from 16 in v8) — 4× more proxy signal


def tokenize(text: str) -> List[str]:
    if not isinstance(text, str):
        return []
    return text.strip().split()


def text_to_indices(text: str, word2idx: dict, max_len: int = 35) -> List[int]:
    tokens  = tokenize(text)[:max_len]
    indices = [word2idx.get(tok, UNK_IDX) for tok in tokens]
    return indices + [PAD_IDX] * (max_len - len(indices))


def prepare_sold_inputs(
    texts: List[str],
    word2idx: dict,
    seq_len: int = 35,
    n_segments: int = 64,
    seed: int = 42
) -> torch.Tensor:
    """
    Sample n_segments tweets → fixed tensor (seq_len, n_segments).
    Kept constant across all HC evaluations so scores are comparable.
    """
    rng         = np.random.RandomState(seed)
    valid_texts = [t for t in texts if isinstance(t, str) and len(t.strip()) > 0]
    sampled_idx = rng.choice(len(valid_texts), size=n_segments, replace=False)
    segments    = [
        torch.tensor(text_to_indices(valid_texts[i], word2idx, seq_len),
                     dtype=torch.long).unsqueeze(1)
        for i in sampled_idx
    ]
    return torch.cat(segments, dim=1).to(DEVICE)


fixed_inputs = prepare_sold_inputs(
    train_texts, word2idx,
    seq_len=SEQ_LEN, n_segments=N_SEGMENTS, seed=SEED
)
print(f'Fixed input tensor shape : {fixed_inputs.shape}')   # (35, 64)
print(f'These {N_SEGMENTS} Sinhala tweets are constant across all HC evaluations')

## 4. Load NAS-Bench-NLP Search Space

In [ ]:
logs_path = os.path.join(NASBENCH_PATH, 'train_logs_single_run', 'logs')
if not os.path.exists(logs_path):
    raise FileNotFoundError(
        f'Logs not found at {logs_path}.\n'
        f'Please run: unzip train_logs_single_run/logs.zip -d train_logs_single_run/'
    )

env        = Environment(logs_path)
search_set = env.get_precomputed_recepies()
print(f'Benchmark search space : {len(search_set)} architectures')

features_path = os.path.join(NASBENCH_PATH, 'data', 'doc2vec_features.csv')
df  = pd.read_csv(features_path).set_index('recepie_id')
ids = env.get_recepie_ids()
X   = df.loc[ids].values   # (14322, 50)

print(f'Doc2Vec feature matrix : {X.shape}')
print(f'Stage 2 mutations will go BEYOND these {len(search_set)} architectures')

## 5. Hidden Covariance Proxy

In [ ]:
NHID            = 200
THEORETICAL_MAX = -float(NHID)   # -200.0


def get_hidden_covariance(model: AWDRNNModel, inputs: torch.Tensor) -> float:
    """
    Hidden Covariance score — exact match to Serianni & Kalita (2023).

    Steps:
      1. Forward pass (no gradients needed)
      2. Last-layer hidden states H: (T, B, D)
      3. permute(2,1,0) → (D, B, T), reshape(D, B*T) → numpy
      4. np.corrcoef → (D, D) Pearson correlation matrix
      5. np.linalg.eig → D eigenvalues (take real part)
      6. score = -sum(log(v+k) + 1/(v+k))   k=1e-5

    Less negative = better. Returns -1e9 for invalid/exploding architectures.
    """
    model.eval()
    with torch.no_grad():
        hidden = model.init_hidden(inputs.size(1))
        _, _, raw_outputs, _ = model(inputs, hidden, return_h=True)

    H = raw_outputs[-1]   # (T, B, D)

    if torch.isnan(H).any() or torch.isinf(H).any():
        return -1e9

    H_np = H.permute(2, 1, 0).reshape(NHID, -1).cpu().numpy()   # (D, B*T)

    correlations = np.corrcoef(H_np)   # (D, D)

    if not np.isfinite(correlations).all():
        return -1e9   # dead arch: some hidden unit has zero variance

    # Use eig (not eigh) to match reference exactly — symmetric matrix so
    # imaginary parts will be negligible; take real part.
    v = np.real(np.linalg.eig(correlations)[0])   # (D,)

    k     = 1e-5
    score = -float(np.sum(np.log(v + k) + 1.0 / (v + k)))

    if not np.isfinite(score):
        return -1e9

    return score


def score_recipe(recipe: dict, ntoken: int, inputs: torch.Tensor) -> float:
    """
    Build an untrained AWDRNNModel from recipe and return HC score.
    Works for benchmark and mutated recipes. Returns -1e9 on any error.
    """
    try:
        model = AWDRNNModel(
            rnn_type   = 'CustomRNN',
            ntoken     = ntoken,
            ninp       = NHID,
            nhid       = NHID,
            nlayers    = 1,
            recepie    = json.dumps(recipe),
            dropout    = 0.0,
            tie_weights= False,
            verbose    = False
        ).to(DEVICE)
        return get_hidden_covariance(model, inputs)
    except Exception:
        return -1e9


print(f'Theoretical maximum HC score (D={NHID}): {THEORETICAL_MAX:.1f}')
print(f'Scores close to {THEORETICAL_MAX:.1f} are best. More negative = worse.')
print()
print('Spot-checking 5 benchmark architectures:')
rng_check = np.random.RandomState(0)
for idx in rng_check.choice(len(search_set), 5, replace=False):
    s      = score_recipe(search_set[idx], VOCAB_SIZE, fixed_inputs)
    status = 'VALID' if s > -1e8 else 'INVALID (penalised)'
    gap    = f'{s - THEORETICAL_MAX:+.1f} from max' if s > -1e8 else ''
    print(f'  Arch {idx:5d}: {s:14.4f}  [{status}] {gap}')

## 6. IGWO Algorithm

In [ ]:
class Wolf:
    def __init__(self):
        self.position = None
        self.cost     = np.inf


class LeaderWolf:
    def __init__(self):
        self.alpha = Wolf()
        self.beta  = Wolf()
        self.delta = Wolf()


class IGWO:
    """
    Improved Grey Wolf Optimizer — v10.

    Navigates the 50D Doc2Vec space and maps positions to the nearest
    benchmark architecture via cosine distance.

    v10 fix: random uniform diversity restart when the swarm collapses
    (≥80% wolves snap to the same arch index). The worst half of the
    population is reinitialised with Lévy-perturbed positions while
    the leaders are preserved. This prevents the search from spending
    most of its budget re-evaluating the same 2-3 architectures.
    """

    def __init__(self, lb, ub, max_iter, pop_size, dim):
        self.lb       = lb
        self.ub       = ub
        self.max_iter = max_iter
        self.pop_size = pop_size
        self.dim      = dim
        self.population  = []
        self.global_best = LeaderWolf()
        self.objective   = None

        # Adaptive weight hyperparameters (unchanged from v8)
        self.alpha_growth_factor = 3.0
        self.delta_growth_factor = 2.0
        self.maximum_weight      = 2.2
        self.minimum_weight      = 0.02

        # Diversity restart threshold
        self.diversity_threshold = 0.80   # restart if ≥80% wolves share one arch

    def set_objective(self, fn):
        self.objective = fn

    def _update_leaders(self):
        self.population.sort(key=lambda w: w.cost)
        candidates = self.population[:3]
        leaders    = [self.global_best.alpha,
                      self.global_best.beta,
                      self.global_best.delta]
        for cand, leader in zip(candidates, leaders):
            if cand.cost < leader.cost:
                leader.position = cand.position.copy()
                leader.cost     = cand.cost

    def _calculate_weights(self, iteration):
        self.alpha_weight = self.maximum_weight * np.exp(
            (iteration / self.max_iter) ** self.alpha_growth_factor *
            np.log(self.minimum_weight / self.maximum_weight)
        )
        self.delta_weight = self.maximum_weight * np.exp(
            (iteration / self.max_iter) ** self.delta_growth_factor *
            np.log(self.minimum_weight / self.maximum_weight)
        )
        self.beta_weight = (self.alpha_weight + self.delta_weight) * 0.5

    def _diversity_restart(self, arch_indices: list):
        """
        v10 fix: when swarm collapses (>=80% wolves on same arch), reinitialise
        the worst half to random uniform positions inside the search bounds.

        v9 used Lévy-perturbed positions around alpha. The problem: Lévy steps
        have heavy tails — after np.clip the restarted wolves pile up at the
        boundary corners of the 50D Doc2Vec space, which are architecturally
        meaningless regions. Random uniform reinit gives true interior diversity
        while still preserving the top half (including leaders) intact.
        """
        counts = Counter(arch_indices)
        most_common_frac = counts.most_common(1)[0][1] / self.pop_size

        if most_common_frac >= self.diversity_threshold:
            self.population.sort(key=lambda w: w.cost)
            half = self.pop_size // 2
            for w in self.population[half:]:
                new_pos  = np.random.uniform(self.lb, self.ub, self.dim)
                new_cost = self.objective(new_pos)
                w.position = new_pos
                w.cost     = new_cost
            return True
        return False

    def run(self):
        # Initialise population
        self.population = [Wolf() for _ in range(self.pop_size)]
        for w in self.population:
            w.position = np.random.uniform(self.lb, self.ub, self.dim)
            w.cost     = self.objective(w.position)
        self._update_leaders()

        for iteration in range(1, self.max_iter + 1):
            self._calculate_weights(iteration)

            for w in self.population:
                r1 = np.random.rand(3)
                r2 = np.random.rand(3)
                A  = 2 * r1 - 1
                C  = 2 * r2

                D_alpha = np.abs(C[0] * self.global_best.alpha.position - w.position)
                X1      = self.global_best.alpha.position - A[0] * D_alpha * self.alpha_weight

                D_beta  = np.abs(C[1] * self.global_best.beta.position - w.position)
                X2      = self.global_best.beta.position - A[1] * D_beta * self.beta_weight

                D_delta = np.abs(C[2] * self.global_best.delta.position - w.position)
                X3      = self.global_best.delta.position - A[2] * D_delta * self.delta_weight

                new_pos  = np.clip((X1 + X2 + X3) / 3.0, self.lb, self.ub)
                new_cost = self.objective(new_pos)

                if new_cost < w.cost:
                    w.position = new_pos
                    w.cost     = new_cost

            self._update_leaders()

            # Diversity restart every 5 iterations.
            # Bug fix vs v9-initial: do NOT build arch_indices inside the wolf
            # loop above (that list was immediately overwritten here anyway,
            # wasting 2,240 cdist calls across 8 rounds). Build it once here,
            # using self._X_ref — not bare X from outer scope.
            if iteration % 5 == 0 and hasattr(self, '_X_ref'):
                arch_indices = [
                    int(np.argmin(distance.cdist([w.position], self._X_ref, 'cosine')))
                    for w in self.population
                ]
                if self._diversity_restart(arch_indices):
                    self._update_leaders()

        return self.global_best.alpha


print('IGWO class defined successfully (v10: uniform diversity restart)')

## 7. Stage 1 — IGWO Global Search

In [ ]:
def run_igwo_round(
    search_set: List,
    X: np.ndarray,
    fixed_inputs: torch.Tensor,
    ntoken: int,
    seed: int,
    budget: int,
    n_pop: int = 20
) -> Tuple[int, float, dict]:
    """
    Run one round of IGWO over the benchmark search space.
    Returns: (best_arch_index, best_score, score_cache)
    """
    np.random.seed(seed)
    torch.manual_seed(seed)

    max_iter    = max(1, (budget - n_pop) // n_pop)
    score_cache = {}   # keyed by arch index — avoids re-evaluating same arch

    def objective(position):
        idx = int(np.argmin(distance.cdist([position], X, 'cosine')))
        if idx in score_cache:
            return -score_cache[idx]
        score            = score_recipe(search_set[idx], ntoken, fixed_inputs)
        score_cache[idx] = score
        return -score   # IGWO minimises — negate to maximise HC

    igwo      = IGWO(
        lb       = X.min(axis=0),
        ub       = X.max(axis=0),
        max_iter = max_iter,
        pop_size = n_pop,
        dim      = X.shape[1]
    )
    igwo._X_ref = X   # give IGWO access to X for diversity restart
    igwo.set_objective(objective)
    igwo.run()

    best_idx   = max(score_cache, key=score_cache.get)
    best_score = score_cache[best_idx]
    return best_idx, best_score, score_cache


# ── Run Stage 1 ────────────────────────────────────────────────────
N_ROUNDS = 8     # v9: 8 rounds (up from 5)
BUDGET   = 300   # v9: 300 budget per round (up from 100)
N_POP    = 20    # v9: 20 wolves (up from 10)

stage1_results  = []
all_score_cache = {}

print('STAGE 1: IGWO Global Search (v9)')
print(f'Rounds: {N_ROUNDS} | Budget per round: {BUDGET} | Wolves: {N_POP}')
print(f'Search space: {len(search_set)} benchmark architectures')
print(f'Proxy: Eigenspectrum HC on {N_SEGMENTS} SOLD tweets\n')

for seed in tqdm(range(N_ROUNDS), desc='IGWO Rounds'):
    best_idx, best_score, cache = run_igwo_round(
        search_set, X, fixed_inputs,
        ntoken=VOCAB_SIZE, seed=seed,
        budget=BUDGET, n_pop=N_POP
    )
    stage1_results.append((best_idx, best_score))
    all_score_cache.update(cache)
    gap = best_score - THEORETICAL_MAX
    print(f'  Round {seed} → Arch {best_idx:5d} | HC = {best_score:.4f} '
          f'(gap from max: {gap:+.1f}) | '
          f'Unique archs evaluated: {len(set(cache.keys()))}')

stage1_best_idx    = max(all_score_cache, key=all_score_cache.get)
stage1_best_score  = all_score_cache[stage1_best_idx]
stage1_best_recipe = search_set[stage1_best_idx]

print(f'\nStage 1 Global Best → Arch {stage1_best_idx} | HC = {stage1_best_score:.4f} '
      f'(gap from max: {stage1_best_score - THEORETICAL_MAX:+.1f})')
print(f'Total unique architectures evaluated: {len(all_score_cache)}')

## 8. Select Top-K Seeds for Stage 2

In [ ]:
K_SEEDS = 8   # v9: 8 seeds (up from 5)

valid_cache  = {k: v for k, v in all_score_cache.items() if v > -1e8}
sorted_archs = sorted(valid_cache.items(), key=lambda x: x[1], reverse=True)
top_k_seeds  = [
    {'index': idx, 'hc_score': score, 'recipe': search_set[idx]}
    for idx, score in sorted_archs[:K_SEEDS]
]

print(f'Top-{K_SEEDS} seed architectures for Stage 2:')
print('=' * 70)
for i, s in enumerate(top_k_seeds):
    nodes = list(s['recipe'].keys())
    gap   = s['hc_score'] - THEORETICAL_MAX
    print(f'  Seed {i+1}: Arch {s["index"]:5d} | '
          f'HC = {s["hc_score"]:8.4f} (gap={gap:+.1f}) | '
          f'Nodes: {nodes}')
print('=' * 70)
print(f'\nThese recipes will be structurally mutated in Stage 2.')

## 9. Recipe Mutation Engine (Stage 2)

In [ ]:
ACTIVATION_OPS  = ['activation_tanh', 'activation_sigm', 'activation_leaky_relu']
COMBINATION_OPS = ['elementwise_sum', 'elementwise_prod', 'blend']
# All h_prev_N nodes — must be included or multi-hidden-state arch validation breaks
ROOT_NODES      = ['x', 'h_prev_0', 'h_prev_1', 'h_prev_2']


def is_valid_recipe(recipe: dict) -> bool:
    """
    Structural validity check — carries all bug fixes from v8.

    Constraints:
      1. At least one h_new_* node (has a hidden output)
      2. At least one activation op (prevents pure linear recurrences that explode
         over long sequences — W^128 diverges if any eigenvalue > 1)
      3. All node inputs reference existing nodes (no dangling references)
      4. No self-references
      5. activation_* → exactly 1 input
      6. elementwise_sum / elementwise_prod → exactly 2 inputs  (v7 bug: was grouped with blend)
      7. blend → exactly 3 inputs             (v7 bug: was grouped with sum/prod as "2 inputs")
      8. linear → at least 1 input
    """
    if not recipe:
        return False
    if not any(k.startswith('h_new') for k in recipe.keys()):
        return False
    if not any(v.get('op') in ACTIVATION_OPS for v in recipe.values()):
        return False
    all_nodes = set(ROOT_NODES) | set(recipe.keys())
    for node_name, node_def in recipe.items():
        if not isinstance(node_def, dict):
            return False
        if 'op' not in node_def or 'input' not in node_def:
            return False
        if any(inp not in all_nodes for inp in node_def['input']):
            return False
        if node_name in node_def['input']:
            return False
        if node_def['op'] in ACTIVATION_OPS and len(node_def['input']) != 1:
            return False
        if node_def['op'] in ['elementwise_sum', 'elementwise_prod'] and len(node_def['input']) != 2:
            return False
        if node_def['op'] == 'blend' and len(node_def['input']) != 3:
            return False
        if node_def['op'] == 'linear' and len(node_def['input']) < 1:
            return False
    return True


def get_recipe_root_nodes(recipe: dict) -> list:
    h_new_count  = sum(1 for k in recipe if k.startswith('h_new_') and k[6:].isdigit())
    h_prev_nodes = [f'h_prev_{i}' for i in range(max(1, h_new_count))]
    return ['x'] + h_prev_nodes


def get_available_inputs(recipe: dict, node_name: str) -> list:
    root_nodes = get_recipe_root_nodes(recipe)
    node_keys  = list(recipe.keys())
    try:
        idx = node_keys.index(node_name)
    except ValueError:
        return root_nodes
    return root_nodes + node_keys[:idx]


def mutate_swap_activation(
    recipe: dict, rng: np.random.RandomState
) -> Optional[dict]:
    """Change one activation op to a different activation."""
    act_nodes = [k for k, v in recipe.items() if v['op'] in ACTIVATION_OPS]
    if not act_nodes:
        return None
    new_r  = copy.deepcopy(recipe)
    target = rng.choice(act_nodes)
    new_r[target]['op'] = str(rng.choice(
        [op for op in ACTIVATION_OPS if op != new_r[target]['op']]
    ))
    return new_r


def mutate_swap_combination(
    recipe: dict, rng: np.random.RandomState
) -> Optional[dict]:
    """
    Change one binary combination op (sum ↔ prod).
    Blend has no swap partner (arity 3) so is excluded.
    """
    BINARY_OPS   = ['elementwise_sum', 'elementwise_prod']
    binary_nodes = [k for k, v in recipe.items() if v['op'] in BINARY_OPS]
    if not binary_nodes:
        return None
    new_r  = copy.deepcopy(recipe)
    target = rng.choice(binary_nodes)
    new_r[target]['op'] = str(rng.choice(
        [op for op in BINARY_OPS if op != new_r[target]['op']]
    ))
    return new_r


def mutate_add_input(
    recipe: dict, rng: np.random.RandomState
) -> Optional[dict]:
    """Add an extra input connection to a linear node."""
    lin_nodes = [k for k, v in recipe.items() if v['op'] == 'linear']
    if not lin_nodes:
        return None
    new_r     = copy.deepcopy(recipe)
    target    = str(rng.choice(lin_nodes))
    available = get_available_inputs(new_r, target)
    new_opts  = [n for n in available if n not in new_r[target]['input']]
    if not new_opts:
        return None
    new_r[target]['input'].append(str(rng.choice(new_opts)))
    return new_r


def mutate_remove_input(
    recipe: dict, rng: np.random.RandomState
) -> Optional[dict]:
    """Remove one input from a linear node with 3+ inputs."""
    lin_nodes = [
        k for k, v in recipe.items()
        if v['op'] == 'linear' and len(v['input']) > 2
    ]
    if not lin_nodes:
        return None
    new_r  = copy.deepcopy(recipe)
    target = str(rng.choice(lin_nodes))
    remove = str(rng.choice(new_r[target]['input']))
    new_r[target]['input'].remove(remove)
    return new_r


def mutate_rewire_input(
    recipe: dict, rng: np.random.RandomState
) -> Optional[dict]:
    """
    Replace one input reference with a different valid input.
    Universal fallback — works on any recipe that has any wiring alternatives.
    Preserves exact node arity so all op constraints are automatically satisfied.
    """
    rewire_candidates = []
    for node_name, node_def in recipe.items():
        avail = get_available_inputs(recipe, node_name)
        if node_def['op'] in ACTIVATION_OPS:
            alternatives = [a for a in avail if a != node_def['input'][0]]
            if alternatives:
                rewire_candidates.append((node_name, 0, alternatives))
        else:
            current_inputs = node_def['input']
            for inp_idx, inp in enumerate(current_inputs):
                alternatives = [
                    a for a in avail
                    if a != inp and a not in current_inputs
                ]
                if alternatives:
                    rewire_candidates.append((node_name, inp_idx, alternatives))
    if not rewire_candidates:
        return None
    node_name, inp_idx, alternatives = rewire_candidates[
        rng.randint(0, len(rewire_candidates))
    ]
    new_r = copy.deepcopy(recipe)
    new_r[node_name]['input'][inp_idx] = str(rng.choice(alternatives))
    return new_r


def mutate_insert_activation(
    recipe: dict, rng: np.random.RandomState
) -> Optional[dict]:
    """
    Insert a new activation node on an existing edge.
    Always creates a NEW node — the only mutation that escapes fully saturated
    architectures where add/remove/rewire all return None.
    Does not insert before an existing activation node (redundant double-activation).
    """
    candidates = []
    for node_name, node_def in recipe.items():
        for inp_idx, inp in enumerate(node_def['input']):
            if inp in recipe and recipe[inp]['op'] in ACTIVATION_OPS:
                continue
            candidates.append((node_name, inp_idx, inp))

    if not candidates:
        return None

    node_name, inp_idx, inp_node = candidates[rng.randint(0, len(candidates))]
    act_op = str(rng.choice(ACTIVATION_OPS))

    existing_nums = [
        int(k.replace('node_', '')) for k in recipe.keys()
        if k.startswith('node_') and k.replace('node_', '').isdigit()
    ]
    new_num  = max(existing_nums) + 1 if existing_nums else 50
    new_name = f'node_{new_num}'

    rebuilt = {}
    for k, v in recipe.items():
        if k == node_name:
            rebuilt[new_name] = {'op': act_op, 'input': [str(inp_node)]}
            new_inputs = [
                new_name if i == inp_idx else str(inp_val)
                for i, inp_val in enumerate(v['input'])
            ]
            rebuilt[k] = {'op': v['op'], 'input': new_inputs}
        else:
            rebuilt[k] = {'op': v['op'], 'input': [str(i) for i in v['input']]}

    return rebuilt if is_valid_recipe(rebuilt) else None


MUTATION_FUNCTIONS = [
    ('swap_activation',         mutate_swap_activation),
    ('swap_combination',        mutate_swap_combination),
    ('add_input_connection',    mutate_add_input),
    ('remove_input_connection', mutate_remove_input),
    ('rewire_input',            mutate_rewire_input),
    ('insert_activation',       mutate_insert_activation),
]

print(f'Mutation engine ready with {len(MUTATION_FUNCTIONS)} mutation types:')
for name, _ in MUTATION_FUNCTIONS:
    print(f'  - {name}')

print(f'\nValidation test on Stage 1 best (Arch {stage1_best_idx}):')
rng_val = np.random.RandomState(0)
all_none = True
for name, fn in MUTATION_FUNCTIONS:
    result = fn(stage1_best_recipe, rng_val)
    valid  = is_valid_recipe(result) if result is not None else False
    status = 'OK' if valid else ('None' if result is None else 'Invalid')
    print(f'  {name}: {status}')
    if valid:
        all_none = False
if all_none:
    print('  WARNING: All mutations failed — check recipe structure')
else:
    print('  At least one mutation succeeded — Stage 2 can proceed')

## 10. Stage 2 — Stagnation-Aware Local Structural Mutation Search

In [ ]:
N_MUTATION_ROUNDS  = 20   # v9: 20 rounds (up from 5)
N_MUTATIONS_PER_OP = 10   # v9: 10 per type per round (up from 5)
EPSILON            = 5.0  # v9: sideways move tolerance on the HC scale
STAGNATION_LIMIT   = 4    # v9: restart after this many stagnant rounds

# Total candidate evaluations per seed = 20 rounds × 6 types × 10 = 1,200
# Total across 8 seeds                 = 9,600 HC evaluations


def run_local_search(
    seed_recipe: dict,
    seed_score: float,
    seed_label: str,
    ntoken: int,
    fixed_inputs: torch.Tensor,
    n_rounds: int   = N_MUTATION_ROUNDS,
    n_per_op: int   = N_MUTATIONS_PER_OP,
    epsilon: float  = EPSILON,
    stagnation_lim: int = STAGNATION_LIMIT,
    rng_seed: int   = 0
) -> Tuple[dict, float, List[dict]]:
    """
    Stagnation-aware greedy local search by structural mutation.

    Each round:
      1. Generate n_per_op candidates per mutation type → score all with HC
      2. Accept the best candidate if it is within epsilon of current score
         (sideways move) or strictly better (improving move)
      3. Increment stagnation_count if no real improvement (< epsilon gain)
      4. After stagnation_limit stagnant rounds → restart from best_ever
       (v10 fix: was seed_recipe in v9, wasting rounds re-exploring old territory)

    Global best is tracked separately — restarts always continue from best_ever.
    Returns: (best_ever_recipe, best_ever_score, search_log)
    """
    rng = np.random.RandomState(rng_seed)

    # current = recipe being mutated this trajectory
    current       = copy.deepcopy(seed_recipe)
    current_score = seed_score

    # best_ever = global best across all rounds and restarts
    best_ever        = copy.deepcopy(seed_recipe)
    best_ever_score  = seed_score

    stagnation_count = 0
    restart_count    = 0
    search_log       = []

    print(f'\n  Seed: {seed_label} | Initial HC = {seed_score:.4f} '
          f'(gap from max: {seed_score - THEORETICAL_MAX:+.1f})')

    for rnd in range(n_rounds):
        # ── Generate candidates ──────────────────────────────────────
        candidates = []
        for mut_name, mut_fn in MUTATION_FUNCTIONS:
            for _ in range(n_per_op):
                candidate = mut_fn(current, rng)
                if (
                    candidate is not None
                    and is_valid_recipe(candidate)
                    and candidate != current
                ):
                    candidates.append((candidate, mut_name))

        if not candidates:
            print(f'    Round {rnd+1:2d}: No valid mutations — stopping early')
            break

        # ── Score all candidates ──────────────────────────────────────
        scored = [
            (cand, score_recipe(cand, ntoken, fixed_inputs), m_name)
            for cand, m_name in candidates
        ]
        best_cand, best_hc, best_mut = max(scored, key=lambda x: x[1])

        real_improvement = best_hc - current_score

        # ── Update best_ever ─────────────────────────────────────────
        if best_hc > best_ever_score:
            best_ever       = best_cand
            best_ever_score = best_hc

        # ── Accept move (strict improvement or epsilon sideways) ──────
        if best_hc >= current_score - epsilon:
            if real_improvement > epsilon:
                # Real improvement — reset stagnation
                stagnation_count = 0
                print(f'    Round {rnd+1:2d}: ✓ HC {current_score:.4f} → '
                      f'{best_hc:.4f} (+{real_improvement:.2f}) via {best_mut}')
            else:
                # Sideways move — increment stagnation
                stagnation_count += 1
                print(f'    Round {rnd+1:2d}: → sideways HC {current_score:.4f} → '
                      f'{best_hc:.4f} ({real_improvement:+.2f}) via {best_mut} '
                      f'[stagnation {stagnation_count}/{stagnation_lim}]')
            current       = best_cand
            current_score = best_hc
        else:
            stagnation_count += 1
            print(f'    Round {rnd+1:2d}: ✗ No improvement '
                  f'(best={best_hc:.4f}) '
                  f'[stagnation {stagnation_count}/{stagnation_lim}]')

        search_log.append({
            'round'            : rnd + 1,
            'n_candidates'     : len(candidates),
            'best_mutation'    : best_mut,
            'best_hc'          : best_hc,
            'current_hc'       : current_score,
            'best_ever_hc'     : best_ever_score,
            'real_improvement' : real_improvement,
            'stagnation_count' : stagnation_count,
            'restart_count'    : restart_count
        })

        # ── Stagnation restart ────────────────────────────────────────
        if stagnation_count >= stagnation_lim:
            restart_count   += 1
            stagnation_count = 0
            # v10 fix: restart from best_ever, not seed_recipe.
            # v9 restarted from seed_recipe/seed_score, discarding all
            # progress made since the search began. With 20 rounds and a
            # 4-round stagnation limit, this wasted up to 12+ rounds
            # re-exploring territory already visited. Restarting from
            # best_ever ensures the search always continues from the
            # best known point and explores outward from there.
            current          = copy.deepcopy(best_ever)
            current_score    = best_ever_score
            print(f'    ↺  Restart #{restart_count} from best_ever '
                  f'(HC={best_ever_score:.4f}, gap={best_ever_score - THEORETICAL_MAX:+.1f})')

    gap = best_ever_score - THEORETICAL_MAX
    print(f'  → Best: {best_ever_score:.4f} '
          f'(gap from max: {gap:+.1f}, '
          f'restarts: {restart_count})')

    return best_ever, best_ever_score, search_log


# ── Run Stage 2 for all top-K seeds ───────────────────────────────
print('STAGE 2: Stagnation-Aware Local Structural Mutation Search (v10)')
print(f'Seeds: {K_SEEDS} | Rounds/seed: {N_MUTATION_ROUNDS} | '
      f'Mutations/type/round: {N_MUTATIONS_PER_OP}')
print(f'Epsilon (sideways): {EPSILON} | Stagnation limit: {STAGNATION_LIMIT}')
print(f'These architectures go BEYOND the {len(search_set)} benchmark architectures\n')

stage2_results = []

for i, seed_arch in enumerate(top_k_seeds):
    best_recipe, best_score, log = run_local_search(
        seed_recipe  = seed_arch['recipe'],
        seed_score   = seed_arch['hc_score'],
        seed_label   = f'Arch {seed_arch["index"]} (HC={seed_arch["hc_score"]:.4f})',
        ntoken       = VOCAB_SIZE,
        fixed_inputs = fixed_inputs,
        n_rounds     = N_MUTATION_ROUNDS,
        n_per_op     = N_MUTATIONS_PER_OP,
        rng_seed     = i
    )
    stage2_results.append({
        'seed_index'  : seed_arch['index'],
        'seed_score'  : seed_arch['hc_score'],
        'final_recipe': best_recipe,
        'final_score' : best_score,
        'improved'    : best_score > seed_arch['hc_score'],
        'improvement' : best_score - seed_arch['hc_score'],
        'log'         : log
    })

print('\nStage 2 complete.')

## 11. Final Selection — Best Architecture Across Both Stages

In [ ]:
print('=' * 70)
print('  STAGE 2 RESULTS SUMMARY (v9)')
print('=' * 70)
print(f'{"Seed Arch":>10} {"Seed HC":>12} {"Final HC":>12} {"Δ HC":>10} '
      f'{"Gap→max":>10} {"Improved?":>10}')
print('-' * 70)

for r in stage2_results:
    gap = r['final_score'] - THEORETICAL_MAX
    print(
        f'{r["seed_index"]:>10} '
        f'{r["seed_score"]:>12.4f} '
        f'{r["final_score"]:>12.4f} '
        f'{r["improvement"]:>+10.4f} '
        f'{gap:>+10.1f} '
        f'{"✓" if r["improved"] else "✗":>10}'
    )
print('=' * 70)

stage2_best = max(stage2_results, key=lambda r: r['final_score'])

print(f'\nStage 1 best : Arch {stage1_best_idx} | HC = {stage1_best_score:.4f} (benchmark)')
print(f'Stage 2 best : Mutated from Arch {stage2_best["seed_index"]} | '
      f'HC = {stage2_best["final_score"]:.4f} (new arch)')

if stage2_best['final_score'] > stage1_best_score:
    print(f'\n✓ Stage 2 found a BETTER architecture beyond the benchmark!')
    print(f'  Improvement over Stage 1 best: +{stage2_best["final_score"] - stage1_best_score:.4f} HC')
    final_recipe = stage2_best['final_recipe']
    final_score  = stage2_best['final_score']
    final_source = f'mutation_of_{stage2_best["seed_index"]}'
    is_mutated   = True
else:
    print(f'\n→ Stage 1 benchmark architecture remains best.')
    final_recipe = stage1_best_recipe
    final_score  = stage1_best_score
    final_source = f'benchmark_{stage1_best_idx}'
    is_mutated   = False

has_act = any(v.get('op') in ACTIVATION_OPS for v in final_recipe.values())
print(f'\nFinal recipe nodes : {list(final_recipe.keys())}')
print(f'Final HC score     : {final_score:.4f}')
print(f'Gap from max (-200): {final_score - THEORETICAL_MAX:+.1f}')
print(f'Source             : {final_source}')
print(f'Has activation ops : {has_act}')

if not has_act:
    print('  WARNING: No activation ops — is_valid_recipe should have caught this.')
    print('  Do not train this architecture.')

## 12. Save Results

In [ ]:
best_stage2_round = max(
    enumerate(stage2_results, start=1),
    key=lambda x: x[1]['final_score']
)
found_in_round = best_stage2_round[0]

output = {
    'best_architecture': {
        # Fields consumed by notebook 03
        'index'            : stage1_best_idx,
        'found_in_round'   : found_in_round,
        'recipe'           : final_recipe,
        'hidden_covariance': final_score,
        # Provenance
        'source'           : final_source,
        'is_mutated'       : is_mutated,
        'version'          : 'v10',
    },
    'stage1': {
        'best_index'  : stage1_best_idx,
        'best_score'  : stage1_best_score,
        'n_evaluated' : len(all_score_cache),
        'config'      : {
            'n_rounds' : N_ROUNDS,
            'budget'   : BUDGET,
            'n_pop'    : N_POP,
        }
    },
    'stage2': [
        {
            'seed_index' : r['seed_index'],
            'seed_score' : r['seed_score'],
            'final_score': r['final_score'],
            'improved'   : r['improved'],
            'improvement': r['improvement'],
            'log'        : r['log']
        }
        for r in stage2_results
    ],
    'search_config': {
        'version'              : 'v10',
        'n_segments'           : N_SEGMENTS,
        'stage1_rounds'        : N_ROUNDS,
        'stage1_budget'        : BUDGET,
        'stage1_n_pop'         : N_POP,
        'stage2_top_k'         : K_SEEDS,
        'stage2_mut_rounds'    : N_MUTATION_ROUNDS,
        'stage2_mut_per_op'    : N_MUTATIONS_PER_OP,
        'stage2_epsilon'       : EPSILON,
        'stage2_stagnation_lim': STAGNATION_LIMIT,
        'n_mutation_types'     : len(MUTATION_FUNCTIONS),
        'mutation_types'       : [n for n, _ in MUTATION_FUNCTIONS],
        'proxy'                : 'hidden_covariance_eigenspectrum',
        'proxy_formula'        : '-sum(log(v+k) + 1/(v+k)) over D×D corrcoef eigenvalues',
        'data'                 : f'SOLD_train_{N_SEGMENTS}tweets_seq{SEQ_LEN}',
        'benchmark_constrained': False
    }
}

out_path = os.path.join(RESULTS_PATH, 'best_architecture.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'Saved to {out_path}')
print(f'\nArchitecture summary (v10):')
print(f'  Benchmark seed  : Arch {stage1_best_idx}')
print(f'  Found in round  : {found_in_round}')
print(f'  Final HC score  : {final_score:.4f}')
print(f'  Gap from max    : {final_score - THEORETICAL_MAX:+.1f}')
print(f'  Is mutated      : {is_mutated}')
print(f'  Has activation  : {has_act}')
print(f'  Recipe nodes    : {list(final_recipe.keys())}')
print(f'\nNotebook 03 is ready to run.')